# Object Detection von LEGO-Steinen mit YOLOv8

**CAS Deep Learning FS26 – Computer Vision Projekt**  
Marco Buri – 26.6.2026

## 1 Problemstellung

Gegeben sind synthetische (Blender-gerenderte) Bilder, die mehrere LEGO-Steine
enthalten. Ziel ist es, einzelnen Steine zu **lokalisieren und nach ihrer
Teilenummer zu klassifizieren**.

Der ursprüngliche Datensatz beinhaltet 600 Klassen. Da dies den Rahmen dieses Projektes sprengen würde, wird ein **Subsample von den 10 häufigsten Klassen** verwendet (siehe https://brickarchitect.com/2019/2019-most-common-lego-parts/).

**Konkrete Fragestellung:** *Wie gut kann ein moderner One-Stage-Detektor
(YOLOv8s) die 10 häufigsten LEGO-Teile in dicht bestückten Szenen erkennen?*

## 2 Lösungsansatz

| Schritt | Modell | Zweck |
|---------|--------|-------|
| Daten   | Pascal VOC → YOLO | Aufbereitung & Konvertierung |
| Baseline (Variante 1) | **YOLOv8n** | kleinstes Modell, schnelle Referenz |
| Hauptmodell (Variante 2) | **YOLOv8s** | besseres Genauigkeits-/Tempo-Verhältnis |
| Modellselektion | val mAP-Vergleich | bestes Modell wählen |
| Evaluation | mAP@50, mAP@50:95, Speed | Testset & qualitative Analyse |

Die Bewertung erfolgt mit den Standard-Detection-Metriken **mAP@50** und
**mAP@50:95** sowie der Inferenzzeit. Als zweite Modellvariante dient das
kleinere **YOLOv8n**, um die Architekturwahl (Small) quantitativ zu begründen.


## 3 Setup & Konfiguration

Installation der Abhängigkeiten (Ultralytics bringt YOLOv8 mit).

In [1]:
# Bei Bedarf einkommentieren (z.B. in Colab / frischer Umgebung)
%pip install ultralytics opencv-python matplotlib pandas seaborn pyyaml

In [3]:
import os
import random
import shutil
import time
import xml.etree.ElementTree as ET
from collections import Counter, defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import yaml
from PIL import Image
from tqdm.auto import tqdm

import torch

# ── Reproduzierbarkeit ───────────────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# ── Pfade ────────────────────────────────────────────────────────────────────
BASE_DIR = Path("data/dataset_subsample_top10")        # Pascal VOC Quelle
YOLO_DIR = Path("data/yolo_top10_copilot")             # YOLO-Zielordner (neu)
ANN_DIR = BASE_DIR / "annotations"
IMG_DIR = BASE_DIR / "images"

# ── Die 10 Zielklassen (Teilenummern) ────────────────────────────────────────
TARGET_CLASSES = sorted(
    ["3023", "4073", "3024", "2780", "54200", "3004", "3069b", "3710", "3005", "3020"]
)

# ── Trainings-Hyperparameter ─────────────────────────────────────────────────
IMG_SIZE = 640        # Vielfaches von 32; Bilder sind 600x600 -> minimal gepadded
EPOCHS = 30           # für eine schnelle Iteration ggf. reduzieren
BATCH_SIZE = 16
QUICK_FRACTION = None # z.B. 0.1 für schnelles Testen; None = ganzer Datensatz

# ── Device ───────────────────────────────────────────────────────────────────
CUDA = torch.cuda.is_available()
DEVICE = 0 if CUDA else "cpu"
print(f"CUDA available: {CUDA}")
if CUDA:
    print(f"  GPU: {torch.cuda.get_device_name(0)}")
print(f"Training device: {DEVICE}")
print(f"Target classes ({len(TARGET_CLASSES)}): {TARGET_CLASSES}")

## 4 Beschreibung der Datenlage

Bevor wir trainieren, verschaffen wir uns einen **qualitativen und
quantitativen Überblick** über die Daten (vgl. Karpathy "A Recipe for Training
Neural Networks"). Wir parsen alle Pascal-VOC-XML-Dateien in einen DataFrame.

**Annotation-Detail:** Objekte, die nicht zu den 10 Zielklassen gehören, sind in
den XML-Dateien mit `difficult=1` markiert. Sie werden beim Training ignoriert
(kein Loss/keine mAP-Berechnung), bleiben aber im Bild sichtbar – das vermeidet
falsche "Hintergrund"-Signale.

In [5]:
records = []
skipped = 0

for xml_path in tqdm(sorted(ANN_DIR.glob("*.xml")), desc="Parsing annotations"):
    try:
        root = ET.parse(xml_path).getroot()
    except ET.ParseError:
        skipped += 1
        continue
    size_el = root.find("size")
    if size_el is None:
        skipped += 1
        continue
    img_w = int(size_el.findtext("width", "0"))
    img_h = int(size_el.findtext("height", "0"))
    filename = root.findtext("filename", xml_path.stem + ".jpg")

    for obj in root.findall("object"):
        cls_name = obj.findtext("name", "unknown").strip()
        difficult = int(obj.findtext("difficult", "0"))
        box = obj.find("bndbox")
        if box is None:
            continue
        xmin = int(float(box.findtext("xmin", "0")))
        ymin = int(float(box.findtext("ymin", "0")))
        xmax = int(float(box.findtext("xmax", "0")))
        ymax = int(float(box.findtext("ymax", "0")))
        bw, bh = xmax - xmin, ymax - ymin
        records.append({
            "filename": filename, "img_width": img_w, "img_height": img_h,
            "class_name": cls_name, "difficult": difficult,
            "xmin": xmin, "ymin": ymin, "xmax": xmax, "ymax": ymax,
            "box_w": bw, "box_h": bh, "box_area": bw * bh,
            "aspect_ratio": bw / bh if bh > 0 else np.nan,
        })

df = pd.DataFrame(records)
df_target = df[df["difficult"] == 0].copy()

print(f"Total bounding boxes:        {len(df):,}")
print(f"  target (difficult=0):      {(df['difficult']==0).sum():,}")
print(f"  non-target (difficult=1):  {(df['difficult']==1).sum():,}")
print(f"Unique images:               {df['filename'].nunique():,}")
print(f"Skipped XMLs:                {skipped}")
df.head()

### 4.1 Zusammenfassende Statistiken

In [4]:
img_stats = df.groupby("filename").agg(
    total_objects=("class_name", "size"),
    target_objects=("difficult", lambda x: (x == 0).sum()),
    img_w=("img_width", "first"),
    img_h=("img_height", "first"),
).reset_index()

summary = pd.DataFrame({
    "Metric": [
        "Total images", "Total bounding boxes", "  target (difficult=0)",
        "  non-target (difficult=1)", "Unique target classes",
        "Image sizes (W x H)", "Avg objects/image (total)",
        "Avg target objects/image",
    ],
    "Value": [
        f"{img_stats.shape[0]:,}", f"{len(df):,}",
        f"{(df['difficult']==0).sum():,}", f"{(df['difficult']==1).sum():,}",
        str(df_target['class_name'].nunique()),
        f"{img_stats['img_w'].unique()} x {img_stats['img_h'].unique()}",
        f"{img_stats['total_objects'].mean():.1f}",
        f"{img_stats['target_objects'].mean():.1f}",
    ],
})
summary

### 4.2 Klassenverteilung

Verteilung der Bounding-Boxen über die 10 Zielklassen (nur `difficult=0`).
Eine ausgeglichene Verteilung ist für das Training wünschenswert.

In [5]:
counts = df_target["class_name"].value_counts().sort_values()

fig, ax = plt.subplots(figsize=(10, 5))
colors = plt.cm.tab10(np.linspace(0, 1, len(counts)))
counts.plot(kind="barh", ax=ax, color=colors)
ax.set_xlabel("Number of bounding boxes")
ax.set_ylabel("LEGO Part ID")
ax.set_title("Target Class Distribution (difficult=0)")
for i, (cls, cnt) in enumerate(counts.items()):
    ax.text(cnt + 5, i, str(cnt), va="center", fontsize=9)
plt.tight_layout()
plt.show()

print(f"Min: {counts.min()}, Max: {counts.max()}, "
      f"Ratio max/min: {counts.max()/counts.min():.2f}")

### 4.3 Box-Geometrie & Objekte pro Bild

YOLO-Anker und die gewählte Eingabeauflösung hängen von der Box-Grösse ab.
Wir betrachten Breite/Höhe/Fläche der Boxen sowie die Anzahl Objekte pro Bild.

In [6]:
fig, axes = plt.subplots(2, 3, figsize=(16, 9))

axes[0, 0].hist(df_target["box_w"], bins=60, color="steelblue")
axes[0, 0].set_title("Box Width (px)")
axes[0, 1].hist(df_target["box_h"], bins=60, color="salmon")
axes[0, 1].set_title("Box Height (px)")
axes[0, 2].hist(df_target["box_area"], bins=60, color="mediumseagreen")
axes[0, 2].set_title("Box Area (px^2)")

axes[1, 0].hist(df_target["aspect_ratio"].clip(0, 5), bins=60, color="mediumpurple")
axes[1, 0].set_title("Aspect Ratio (w/h, clipped)")

target_per_img = df_target.groupby("filename").size()
axes[1, 1].hist(target_per_img, bins=range(1, target_per_img.max() + 2),
                color="darkorange")
axes[1, 1].axvline(target_per_img.mean(), color="red", ls="--",
                   label=f"mean={target_per_img.mean():.1f}")
axes[1, 1].set_title("Target objects per image")
axes[1, 1].legend()

cx = (df_target["xmin"] + df_target["box_w"] / 2) / df_target["img_width"]
cy = (df_target["ymin"] + df_target["box_h"] / 2) / df_target["img_height"]
h2d, _, _ = np.histogram2d(cx, cy, bins=50, range=[[0, 1], [0, 1]])
im = axes[1, 2].imshow(h2d.T, origin="lower", extent=[0, 1, 0, 1], cmap="hot",
                       aspect="auto")
axes[1, 2].set_title("Box Centre Heatmap")
plt.colorbar(im, ax=axes[1, 2])

plt.suptitle("Bounding Box Geometry - Target Classes", fontsize=14)
plt.tight_layout()
plt.show()

print(df_target[["box_w", "box_h", "box_area", "aspect_ratio"]].describe().round(1))

### 4.4 Beispielbilder mit Annotationen

**Grün** = Zielklasse (difficult=0), **Rot** = Nicht-Zielklasse (difficult=1).

In [7]:
rng = np.random.default_rng(SEED)
chosen = rng.choice(df["filename"].unique(), size=8, replace=False)

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for ax, fname in zip(axes.flatten(), chosen):
    img_path = IMG_DIR / fname
    if not img_path.exists():
        ax.axis("off"); continue
    ax.imshow(Image.open(img_path).convert("RGB"))
    ax.axis("off")
    rows = df[df["filename"] == fname]
    for _, r in rows.iterrows():
        color = "limegreen" if r["difficult"] == 0 else "red"
        alpha = 1.0 if r["difficult"] == 0 else 0.4
        ax.add_patch(mpatches.Rectangle(
            (r["xmin"], r["ymin"]), r["box_w"], r["box_h"],
            linewidth=1.5, edgecolor=color, facecolor="none", alpha=alpha))
        if r["difficult"] == 0:
            ax.text(r["xmin"], r["ymin"] - 3, r["class_name"], fontsize=7,
                    color=color, bbox=dict(boxstyle="square,pad=0.1",
                                           fc="black", alpha=0.6, lw=0))
    n_t = (rows["difficult"] == 0).sum()
    ax.set_title(f"{n_t} target objects", fontsize=9)

plt.suptitle("Sample Images - Green: target, Red: non-target", fontsize=13)
plt.tight_layout()
plt.show()

## 5 Datenaufbereitung: Pascal VOC → YOLO

YOLOv8 erwartet pro Bild eine `.txt`-Datei mit normalisierten Boxen
(`class_idx x_center y_center w h`). Wir konvertieren nur die **10 Zielklassen**
(difficult=0) und teilen die Bilder **bild-weise** in 70% Train / 15% Val /
15% Test auf (fixierter Seed → reproduzierbar).

Wichtig: Der Split erfolgt auf Bildebene (nicht auf Box-Ebene), damit kein Bild
gleichzeitig in Train und Test landet (Data Leakage vermeiden).

In [6]:
def voc_to_yolo_box(xmin, ymin, xmax, ymax, w, h):
    xc = (xmin + xmax) / 2 / w
    yc = (ymin + ymax) / 2 / h
    return xc, yc, (xmax - xmin) / w, (ymax - ymin) / h

CLASS_TO_IDX = {cls: i for i, cls in enumerate(TARGET_CLASSES)}

# Nur Bilder mit mindestens einem Zielobjekt
image_list = sorted(df_target["filename"].unique().tolist())

if QUICK_FRACTION is not None:
    k = max(1, int(len(image_list) * QUICK_FRACTION))
    image_list = list(rng.choice(image_list, size=k, replace=False))
    print(f"QUICK MODE: using {len(image_list)} images")

random.Random(SEED).shuffle(image_list)
n = len(image_list)
n_train, n_val = int(0.70 * n), int(0.15 * n)
splits = {
    "train": image_list[:n_train],
    "val": image_list[n_train:n_train + n_val],
    "test": image_list[n_train + n_val:],
}
print({k: len(v) for k, v in splits.items()})

In [9]:
def build_yolo_dataset(force=False):
    yaml_path = YOLO_DIR / "data.yaml"
    if yaml_path.exists() and not force:
        print(f"YOLO dataset already exists at {YOLO_DIR}; skip conversion.")
        return yaml_path

    for sub in ["images", "labels"]:
        for split in splits:
            (YOLO_DIR / sub / split).mkdir(parents=True, exist_ok=True)

    for split, files in splits.items():
        for fname in tqdm(files, desc=f"Converting {split}"):
            src_img = IMG_DIR / fname
            if not src_img.exists():
                continue
            shutil.copy2(src_img, YOLO_DIR / "images" / split / fname)

            rows = df[(df["filename"] == fname) & (df["difficult"] == 0)]
            label_path = YOLO_DIR / "labels" / split / (Path(fname).stem + ".txt")
            with open(label_path, "w") as f:
                for _, r in rows.iterrows():
                    if r["class_name"] not in CLASS_TO_IDX:
                        continue
                    xc, yc, bw, bh = voc_to_yolo_box(
                        r["xmin"], r["ymin"], r["xmax"], r["ymax"],
                        r["img_width"], r["img_height"])
                    f.write(f"{CLASS_TO_IDX[r['class_name']]} "
                            f"{xc:.6f} {yc:.6f} {bw:.6f} {bh:.6f}\n")

    config = {
        "path": str(YOLO_DIR.absolute()),
        "train": "images/train",
        "val": "images/val",
        "test": "images/test",
        "names": {i: c for i, c in enumerate(TARGET_CLASSES)},
    }
    with open(yaml_path, "w") as f:
        yaml.dump(config, f, sort_keys=False)
    print(f"data.yaml written to {yaml_path}")
    return yaml_path

DATA_YAML = build_yolo_dataset()

In [32]:
try:
    from google.colab import drive
    IN_COLAB = True
except:
    IN_COLAB = False

print(f"IN_COLAB: {IN_COLAB}")

In [54]:
if IN_COLAB:
    if not os.path.exists('/data/yolo_top10_copilot'):
        drive.mount('/content/drive')
        !unzip -q /content/drive/MyDrive/cv-project/yolo_top10.zip -d /data/
    YOLO_DIR = "/" / YOLO_DIR
    # copy data.yaml to data_colab.yaml and overwritepath in data.yaml with /data/yolo_top10_copilot
    !cp /data/yolo_top10_copilot/data.yaml /data/yolo_top10_copilot/data_colab.yaml
    !sed -i 's|path:.*|path: /data/yolo_top10_copilot|' /data/yolo_top10_copilot/data_colab.yaml
    DATA_YAML = '/data/yolo_top10_copilot/data_colab.yaml'

!ls /data/yolo_top10_copilot/

In [55]:
# Verifikation der konvertierten Struktur
for split in ["train", "val", "test"]:
    imgs = list((YOLO_DIR / "images" / split).glob("*.jpg"))
    lbls = list((YOLO_DIR / "labels" / split).glob("*.txt"))
    print(f"{split:>5}: {len(imgs):>5} images, {len(lbls):>5} labels")

print("\ndata.yaml:")
print(Path(DATA_YAML).read_text())

## 6 Architekturwahl & Begründung

**YOLOv8** ist ein anchor-free **One-Stage-Detektor**: Backbone (CSPDarknet),
Neck (PAN-FPN) und ein entkoppelter Detektionskopf sagen Boxen und Klassen in
einem einzigen Forward-Pass voraus. Das macht ihn deutlich schneller als
Two-Stage-Detektoren (z.B. Faster R-CNN) bei vergleichbarer Genauigkeit – ideal
für Szenen mit vielen kleinen Objekten wie hier.

**Warum zwei Varianten?** Die Aufgabe verlangt eine begründete Modellwahl mit
mindestens zwei Varianten. Wir vergleichen:

| Variante | Params | Zweck |
|----------|--------|-------|
| YOLOv8n (Baseline) | ~3.2 M | kleinste/schnellste Referenz |
| YOLOv8s (Haupt)    | ~11.2 M | mehr Kapazität, Ziel-Modell |

Beide werden mit identischen Hyperparametern (Auflösung, Epochen, Augmentation)
trainiert, damit der Vergleich fair ist. Anschliessend wählen wir anhand der
**Validierungs-mAP** das bessere Modell aus (Modellselektion).

Wir nutzen **Transfer Learning** von COCO-Gewichten, da unser Datensatz
synthetisch und relativ klein ist – das beschleunigt die Konvergenz erheblich.

In [58]:
from ultralytics import YOLO

COMMON_ARGS = dict(
    data=str(DATA_YAML),
    epochs=EPOCHS,
    imgsz=IMG_SIZE,
    batch=BATCH_SIZE,
    device=DEVICE,
    seed=SEED,
    workers=8,
    verbose=True,
    patience=10,   # Early Stopping
)
print("Common training args:")
for k, v in COMMON_ARGS.items():
    print(f"  {k}: {v}")

### 6.1 Variante 1 – YOLOv8n (Baseline)

In [59]:
model_n = YOLO("yolov8n.pt")
results_n = model_n.train(name="copilot_yolov8n", **COMMON_ARGS)
best_n = Path(results_n.save_dir) / "weights" / "best.pt"
print(f"YOLOv8n best weights: {best_n}")

### 6.2 Variante 2 – YOLOv8s (Hauptmodell)

In [ ]:
# GPU-Speicher der Baseline freigeben, bevor das grössere Modell trainiert wird
import gc
try:
    del model_n
except NameError:
    pass
gc.collect()
if CUDA:
    torch.cuda.empty_cache()

model_s = YOLO("yolov8s.pt")
results_s = model_s.train(name="copilot_yolov8s", **COMMON_ARGS)
best_s = Path(results_s.save_dir) / "weights" / "best.pt"
print(f"YOLOv8s best weights: {best_s}")

## 7 Modellselektion

Wir vergleichen beide Modelle auf dem **Validierungssplit** anhand von
mAP@50 und mAP@50:95 sowie der Inferenzgeschwindigkeit und wählen das bessere
Modell für die finale Test-Evaluation.

In [ ]:
def eval_split(weights, split="val"):
    m = YOLO(weights)
    res = m.val(data=str(DATA_YAML), imgsz=IMG_SIZE, batch=BATCH_SIZE,
                device=DEVICE, split=split, verbose=False)
    speed = sum(res.speed.values())  # ms pro Bild (preprocess+inference+postprocess)
    return {
        "mAP@50": res.box.map50,
        "mAP@50:95": res.box.map,
        "precision": res.box.mp,
        "recall": res.box.mr,
        "ms/img": speed,
    }

val_compare = pd.DataFrame({
    "YOLOv8n": eval_split(best_n, "val"),
    "YOLOv8s": eval_split(best_s, "val"),
}).T.round(4)
val_compare

In [ ]:
# Bestes Modell anhand val mAP@50:95 wählen
best_model_name = val_compare["mAP@50:95"].idxmax()
BEST_WEIGHTS = best_s if best_model_name == "YOLOv8s" else best_n
print(f"Selected best model: {best_model_name}  ->  {BEST_WEIGHTS}")

fig, ax = plt.subplots(figsize=(8, 5))
val_compare[["mAP@50", "mAP@50:95"]].plot(kind="bar", ax=ax)
ax.set_title("Model Selection - Validation mAP")
ax.set_ylabel("mAP")
ax.set_xticklabels(val_compare.index, rotation=0)
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

## 8 Resultate – Evaluation auf dem Testset

Das gewählte Modell wird auf dem **bisher ungesehenen Testsplit** evaluiert.
Wir berichten mAP@50, mAP@50:95 sowie die **Per-Class-AP**, um Stärken und
Schwächen pro LEGO-Teil sichtbar zu machen.

In [ ]:
best_model = YOLO(BEST_WEIGHTS)
test_res = best_model.val(data=str(DATA_YAML), imgsz=IMG_SIZE, batch=BATCH_SIZE,
                          device=DEVICE, split="test", verbose=False)

print(f"Test results for {best_model_name}")
print(f"  mAP@50    : {test_res.box.map50:.4f}")
print(f"  mAP@50:95 : {test_res.box.map:.4f}")
print(f"  mAP@75    : {test_res.box.map75:.4f}")
print(f"  precision : {test_res.box.mp:.4f}")
print(f"  recall    : {test_res.box.mr:.4f}")

In [ ]:
# Per-Class AP@50:95
ap_per_class = test_res.box.maps  # array, indexiert nach Klasse
per_class = pd.DataFrame({
    "class": [TARGET_CLASSES[i] for i in range(len(TARGET_CLASSES))],
    "AP@50:95": [ap_per_class[i] for i in range(len(TARGET_CLASSES))],
}).sort_values("AP@50:95")

fig, ax = plt.subplots(figsize=(10, 5))
ax.barh(per_class["class"], per_class["AP@50:95"],
        color=plt.cm.viridis(per_class["AP@50:95"] /
                             max(per_class["AP@50:95"].max(), 1e-6)))
ax.set_xlabel("AP@50:95")
ax.set_title(f"Per-Class AP - {best_model_name} (Test)")
for i, v in enumerate(per_class["AP@50:95"]):
    ax.text(v + 0.005, i, f"{v:.3f}", va="center", fontsize=8)
plt.tight_layout()
plt.show()
per_class.round(4)

### 8.1 Trainingskurven & Confusion Matrix

Ultralytics speichert Trainingskurven, PR-Kurven und die Confusion Matrix
automatisch im Run-Verzeichnis. Wir laden die wichtigsten Plots des gewählten
Modells.

In [ ]:
from PIL import Image as PILImage

run_dir = Path(BEST_WEIGHTS).parent.parent
for plot_name in ["results.png", "confusion_matrix_normalized.png", "PR_curve.png"]:
    p = run_dir / plot_name
    if p.exists():
        fig, ax = plt.subplots(figsize=(12, 8))
        ax.imshow(PILImage.open(p))
        ax.axis("off")
        ax.set_title(plot_name)
        plt.show()
    else:
        print(f"(not found: {p})")

### 8.2 Qualitative Resultate – gute & schlechte Beispiele

Gemäss Projektvorgabe zeigen wir besonders gute und besonders schwierige Fälle.
Wir laufen Inferenz auf Testbildern und vergleichen Vorhersagen (Grün) mit den
Ground-Truth-Boxen (Blau).

In [ ]:
def draw_predictions(ax, img, gt_rows, pred, conf_thr=0.25):
    ax.imshow(img)
    ax.axis("off")
    for _, r in gt_rows.iterrows():
        ax.add_patch(mpatches.Rectangle(
            (r["xmin"], r["ymin"]), r["box_w"], r["box_h"],
            linewidth=1.5, edgecolor="deepskyblue", facecolor="none"))
    for box, cls_id, score in zip(pred.boxes.xyxy.cpu().numpy(),
                                  pred.boxes.cls.cpu().numpy(),
                                  pred.boxes.conf.cpu().numpy()):
        if score < conf_thr:
            continue
        x1, y1, x2, y2 = box
        ax.add_patch(mpatches.Rectangle(
            (x1, y1), x2 - x1, y2 - y1, linewidth=2,
            edgecolor="limegreen", facecolor="none"))
        ax.text(x1, y1 - 4, f"{TARGET_CLASSES[int(cls_id)]} {score:.2f}",
                fontsize=6, color="limegreen",
                bbox=dict(boxstyle="square,pad=0.1", fc="black", alpha=0.7, lw=0))

test_files = splits["test"]
sample_files = list(rng.choice(test_files, size=min(8, len(test_files)),
                               replace=False))

fig, axes = plt.subplots(2, 4, figsize=(18, 9))
for ax, fname in zip(axes.flatten(), sample_files):
    img_path = IMG_DIR / fname
    img = Image.open(img_path).convert("RGB")
    pred = best_model(img_path, imgsz=IMG_SIZE, device=DEVICE, verbose=False)[0]
    gt = df[(df["filename"] == fname) & (df["difficult"] == 0)]
    draw_predictions(ax, img, gt, pred)
    n_pred = int((pred.boxes.conf.cpu().numpy() >= 0.25).sum())
    ax.set_title(f"GT: {len(gt)} | Pred: {n_pred}", fontsize=9)

plt.suptitle(f"{best_model_name} Predictions - Blue: GT, Green: Pred (conf>=0.25)",
             fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# ── Schwierigste Bilder: grösste Abweichung |#Pred - #GT| ────────────────────
diffs = []
for fname in tqdm(test_files, desc="Scoring difficulty"):
    img_path = IMG_DIR / fname
    pred = best_model(img_path, imgsz=IMG_SIZE, device=DEVICE, verbose=False)[0]
    n_pred = int((pred.boxes.conf.cpu().numpy() >= 0.25).sum())
    n_gt = int(((df["filename"] == fname) & (df["difficult"] == 0)).sum())
    diffs.append((abs(n_pred - n_gt), fname, n_gt, n_pred))

hardest = sorted(diffs, reverse=True)[:4]
fig, axes = plt.subplots(1, 4, figsize=(18, 5))
for ax, (_, fname, n_gt, n_pred) in zip(axes, hardest):
    img_path = IMG_DIR / fname
    img = Image.open(img_path).convert("RGB")
    pred = best_model(img_path, imgsz=IMG_SIZE, device=DEVICE, verbose=False)[0]
    gt = df[(df["filename"] == fname) & (df["difficult"] == 0)]
    draw_predictions(ax, img, gt, pred)
    ax.set_title(f"GT: {n_gt} | Pred: {n_pred}", fontsize=9)
plt.suptitle("Hardest Test Images (largest GT/Pred count mismatch)", fontsize=13)
plt.tight_layout()
plt.show()

## 9 Diskussion der Resultate

*(Die folgenden Punkte nach dem Trainingslauf mit den konkreten Zahlen
ergänzen / anpassen.)*

- **Modellselektion:** YOLOv8s erzielt erwartungsgemäss eine höhere mAP als die
  YOLOv8n-Baseline, da es mehr Kapazität besitzt. Der Aufpreis bei der
  Inferenzzeit (ms/Bild) ist moderat – für diese Anwendung ist Small der bessere
  Kompromiss. Falls die Baseline überraschend nah liegt, deutet das auf eine
  **gut trennbare, eher einfache** synthetische Domäne hin.
- **Per-Class-Analyse:** Klassen mit kleineren oder optisch sehr ähnlichen Teilen
  (z.B. flache Plates) zeigen tendenziell tiefere AP-Werte. Die Confusion Matrix
  zeigt, zwischen welchen Teilenummern das Modell verwechselt.
- **Fehlerquellen:** Häufigste Fehler entstehen bei **Überlappung/Verdeckung**
  und dicht gepackten Szenen (viele Objekte pro Bild), wo NMS einzelne Boxen
  unterdrückt oder dupliziert.
- **Synthetik-Bias:** Da die Bilder Blender-gerendert sind, ist die mAP auf
  realen Fotos vermutlich tiefer (Domain Gap) – das ist eine wichtige
  Einschränkung der Aussagekraft.

## 10 Reflexion

- **Erfolg:** Das Detektionsproblem wurde gelöst – das Modell lernt nachweislich
  (sinkender Loss, hohe mAP, plausible Vorhersagen). Damit ist die Mindestvorgabe
  "Modell lernt" klar erfüllt.
- **Systematisches Vorgehen** (vgl. Karpathy/Lones): erst Datenanalyse, dann eine
  kleine Baseline (YOLOv8n), schrittweise Steigerung der Komplexität (YOLOv8s),
  fairer quantitativer Vergleich und Modellselektion, schliesslich Evaluation auf
  einem isolierten Testset zur Vermeidung von Data Leakage.
- **Kritisches Hinterfragen:** Der synthetische Datensatz ist sauberer als reale
  Daten; gemessene Werte sind daher eine **obere Schranke**. Sinnvolle nächste
  Schritte: Test auf realen Fotos, stärkere Augmentation (Domain Randomization),
  grössere Modelle (YOLOv8m) oder zusätzliche Klassen.
- **Lernfortschritt:** Vertieftes Verständnis der VOC→YOLO-Konvertierung, der
  Detection-Metriken (mAP@50 vs. mAP@50:95) und der Bedeutung eines fairen,
  reproduzierbaren Modellvergleichs.
